# 21 — Direction 1, step 6: train the policy through the replica (SHAC), then score it in the original

One session does everything (`hvac_controller/d1/train.py`, `policy.py`, `evaluate.py`):

1. **smoke** (≈ 5 min): layout of every scenario against its simulator objects, parity of the submission path
   (`KagglePolicy` on the organisers' simulator, CPU) with the training path (GPU), CUDA graph against eager, a short
   run and a 288-step replica score. Stops the notebook if anything fails.
2. **stage 1**: one training run per entry of `STAGE1` (both GPUs, one run each at a time), `MINUTES` each.
3. **stage 2**: the best stage-1 run's settings with each change of `STAGE2`.
4. **original**: the `EVAL_TOP` best runs (by their replica score) for 30 days in the organisers' simulator on
   `EVAL_SCENARIOS`, seeds 0–3, on the session's 4 CPUs, against K10's null group from notebook 20 (z per scenario).

**MODE = "pilot"** (6a): train and score on atlanta_2. **MODE = "held_out"** (6b): train on the 25 scenarios of the
other buildings, select by and score on `HELD_OUT`'s 5 (set `STAGE1` to the pilot's best settings and `MINUTES` to the
budget). **MODE = "all"** (6b.3): train on all 30.

**Setup:** accelerator **GPU T4 ×2**; Internet on (clones the repo); attach the competition data and the K10 dataset
(notebook 20's results, `original__*__k10__*.json`, found anywhere under `/kaggle/input`). Replica caches (`*.pkl`)
found under `/kaggle/input` are used; missing ones are built first on the session's CPUs (`cache.py`, ≈ 2 min for
atlanta_2, ≈ 20–25 min for all 30). Pilot ≈ 2.5–3 h. **Output:** `/kaggle/working/d1_train_<mode>.zip` (logs, reports, the selected
policies `d1_policy.pt`, original-simulator results).

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
MODE = "pilot"            # "pilot", "held_out" or "all"
PILOT = "building_train_1__atlanta_2"
HELD_OUT = "building_train_2"
STAGE1 = [{"h": 36}, {"h": 72}, {"h": 144}, {"h": 288}]
STAGE2 = [{"gamma": 0.99}, {"gamma": 0.998}, {"detach_obs": 1}]   # applied to the best stage-1 run's settings
COMMON = {"eval_minutes": 10}
ROW_STEPS = 2048 * 36     # rows x h per window: 12.2 GB on a T4 for atlanta_2 (pilot smoke); rows = ROW_STEPS // h
MINUTES = 25              # per training run
EVAL_TOP = 2              # runs scored in the original simulator
SEEDS = 4

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

import torch
N_GPU = torch.cuda.device_count()
assert N_GPU >= 1, "no GPU: set the accelerator to GPU T4 x2"
print(torch.__version__, N_GPU, "x", torch.cuda.get_device_name())

REPO = Path("/tmp/hvac-controller")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(COMMIT)
SHIPPED = next(p for p in sorted(Path("/kaggle/input").rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
shutil.copytree(SHIPPED, REPO / "smart_buildings" / "training_scenarios", dirs_exist_ok=True)
sys.path.insert(0, str(REPO))

ROOT = REPO / "smart_buildings" / "training_scenarios"
ALL = sorted(f"{p.parent.parent.name}__{p.parent.name}" for p in ROOT.glob("*/*/config.gin"))
K10 = sorted(Path("/kaggle/input").rglob("original__*__k10__s*.json"))
if MODE == "pilot":
    TRAIN, VAL = [PILOT], []
elif MODE == "held_out":
    TRAIN = [s for s in ALL if not s.startswith(HELD_OUT + "__")]
    VAL = [s for s in ALL if s.startswith(HELD_OUT + "__")]
else:
    TRAIN, VAL = ALL, []
EVAL_SCENARIOS = [s.replace("__", "/", 1) for s in (VAL or TRAIN)]
print(len(K10), "K10 results; train:", TRAIN, "\nheld out:", VAL)

# Replica caches: those attached, the others built now (3 at a time on the CPUs).
CACHES = {p.stem: p for p in sorted(Path("/kaggle/input").rglob("*.pkl"))}
BUILT = Path("/kaggle/temp/d1_caches")
BUILT.mkdir(parents=True, exist_ok=True)
todo = [s for s in TRAIN + VAL if s not in CACHES]
benv = dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
t0, running = time.time(), []
while todo or running:
    while todo and len(running) < 3:
        s = todo.pop(0)
        cfg = ROOT / s.replace("__", "/", 1) / "config.gin"
        running.append((s, subprocess.Popen([sys.executable, "-m", "hvac_controller.d1.cache", str(BUILT / f"{s}.pkl"),
                                             "--config", str(cfg)], cwd=REPO, env=benv, stdout=subprocess.PIPE,
                                            stderr=subprocess.STDOUT, text=True)))
    for item in list(running):
        s, proc = item
        if proc.poll() is not None:
            running.remove(item)
            print(f"[{time.time() - t0:5.0f}s] {s} exit {proc.returncode}: {proc.stdout.read().strip()[-400:]}", flush=True)
    time.sleep(3)
CACHES.update({p.stem: p for p in sorted(BUILT.glob("*.pkl"))})
missing = [s for s in TRAIN + VAL if s not in CACHES]
assert not missing, f"caches missing: {missing}"

OUT = Path(f"/kaggle/working/d1_train_{MODE}")
OUT.mkdir(parents=True, exist_ok=True)
ENV = dict(os.environ, PYTHONPATH=str(REPO))


def train_cmd(out_dir, settings, extra=()):
    cmd = [sys.executable, "-m", "hvac_controller.d1.train", str(out_dir),
           "--caches", ",".join(str(CACHES[s]) for s in TRAIN)]
    if VAL:
        cmd += ["--val-caches", ",".join(str(CACHES[s]) for s in VAL)]
    for k, v in {**COMMON, **settings}.items():
        cmd += [f"--{k.replace('_', '-')}", str(v)]
    return cmd + list(extra)


def tail(path, n=3000):
    text = Path(path).read_text() if Path(path).exists() else ""
    return text[-8000:] if "Traceback" in text else text[-n:]

In [ ]:
# 1. Smoke: start-up checks and a two-minute run on GPU 0. Stops here if anything is wrong.
smoke = OUT / "smoke"
smoke.mkdir(exist_ok=True)
t0 = time.time()
with open(smoke / "log.txt", "w") as log:
    rc = subprocess.run(train_cmd(smoke, {"h": 36, "rows": ROW_STEPS // 36, "minutes": 2, "eval_minutes": 100,
                                          "eval_steps": 288}, ["--smoke"]),
                        cwd=REPO, env=dict(ENV, CUDA_VISIBLE_DEVICES="0"), stdout=log, stderr=subprocess.STDOUT).returncode
print(f"smoke exit {rc} in {time.time() - t0:.0f}s\n" + tail(smoke / "log.txt"))
rep = json.loads((smoke / "report.json").read_text()) if (smoke / "report.json").exists() else {}
assert rc == 0 and rep.get("parity", {}).get("ok"), "smoke failed: read the log above"
print("parity", rep["parity"], "| graph", rep.get("setup", {}).get("graph"))

In [ ]:
# 2-3. Training runs, one per GPU at a time.
def name_of(settings):
    return "_".join(f"{k}{v}" for k, v in settings.items()) or "default"


def run_stage(stage):
    pending, running, done = list(stage), [], {}
    free = list(range(N_GPU))
    t0 = time.time()
    while pending or running:
        while pending and free:
            settings, gpu = pending.pop(0), free.pop(0)
            d = OUT / name_of(settings)
            d.mkdir(exist_ok=True)
            log = open(d / "log.txt", "w")
            rows = settings.get("rows", ROW_STEPS // settings.get("h", 72))
            proc = subprocess.Popen(train_cmd(d, {"rows": rows, **settings, "minutes": MINUTES}), cwd=REPO, stdout=log,
                                    stderr=subprocess.STDOUT, env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)))
            running.append((settings, gpu, proc, log, d))
            print(f"[{time.time() - t0:5.0f}s] GPU {gpu}: {name_of(settings)}", flush=True)
        for item in list(running):
            settings, gpu, proc, log, d = item
            if proc.poll() is not None:
                log.close()
                running.remove(item)
                free.append(gpu)
                r = json.loads((d / "report.json").read_text()) if (d / "report.json").exists() else {}
                done[name_of(settings)] = {"settings": settings, "dir": d, "best": r.get("best", float("inf")),
                                           "report": r, "exit": proc.returncode}
                print(f"[{time.time() - t0:5.0f}s] {name_of(settings)} exit {proc.returncode} best {r.get('best')}\n"
                      + tail(d / "log.txt", 1500), flush=True)
        time.sleep(10)
    return done


RUNS = run_stage(STAGE1)
best1 = min(RUNS.values(), key=lambda r: r["best"])
print("stage 1 best:", name_of(best1["settings"]), best1["best"])
if STAGE2:
    RUNS.update(run_stage([{**best1["settings"], **change} for change in STAGE2]))

In [ ]:
# 4. The best runs in the organisers' simulator against K10's null group.
from hvac_controller.d1 import evaluate as ev

top = sorted((r for r in RUNS.values() if (r["dir"] / "d1_policy.pt").exists()), key=lambda r: r["best"])[:EVAL_TOP]
EVAL = OUT / "original"
EVAL.mkdir(exist_ok=True)
for p in K10:
    shutil.copy(p, EVAL / p.name)
policies = ",".join(f"trained:{r['dir'] / 'd1_policy.pt'}" for r in top)
cmd = [sys.executable, "-m", "hvac_controller.d1.evaluate", str(EVAL), "--scenarios", ",".join(EVAL_SCENARIOS),
       "--scenario-root", str(REPO / "smart_buildings" / "training_scenarios"), "--policies", policies,
       "--seeds", str(SEEDS), "--where", "original", "--n-procs", "4"]
threads = dict(ENV, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", CUDA_VISIBLE_DEVICES="")
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=REPO, env=threads, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
print("exit", proc.wait(), f"in {time.time() - t0:.0f}s")

In [ ]:
# Summary and zip.
results = [json.loads(p.read_text()) for p in sorted(EVAL.glob("*.json"))]
ens = ev.ensembles(results)
print(f"{'run':28s} {'replica best':>12} {'it':>6} {'env-steps/s':>12} graph")
for name, r in sorted(RUNS.items(), key=lambda kv: kv[1]["best"]):
    rep = r["report"]
    speeds = [json.loads(l)["env_steps_per_s"] for l in open(r["dir"] / "train_log.jsonl")] if (r["dir"] / "train_log.jsonl").exists() else [0]
    print(f"{name:28s} {r['best']:12.5f} {rep.get('iterations', 0):6d} {sorted(speeds)[len(speeds) // 2]:12,.0f} "
          f"{rep.get('setup', {}).get('graph')}")
gate = []
for r in top:
    tag = r["dir"].name
    cmp = ev.compare(ens, tag, "k10")
    print(f"\n{tag} in the original vs K10 over {SEEDS} seeds (z = gap / sd of both ensembles, floor 0.5 % of K10):")
    for scenario, c in cmp.items():
        if scenario == "mean":
            print(f"  mean over scenarios: {c['candidate']:.5f} vs K10 {c['null']:.5f}")
            continue
        e = ens.get(("original", scenario, tag), {})
        print(f"  {scenario:30s} {c['candidate']:.5f} vs {c['null']:.5f} ({c['rel']:+.1%}, z {c['z']:+.1f}, sd "
              f"{c['sd_rel']:.2%}) | regret {e.get('regret', 0):.4f} cost {e.get('cost', 0):.4f} carbon {e.get('carbon', 0):.4f}")
        gate.append(c["z"] < -2.0)
print(f"\nGATE: beats K10 beyond its null group (z < -2) on {sum(gate)}/{len(gate)} scenario-runs")
name = f"/kaggle/working/d1_train_{MODE}.zip"
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and p.suffix != ".pt" or p.name == "d1_policy.pt" and any(p.parent == r["dir"] for r in top):
            z.write(p, p.relative_to(OUT.parent))
print("wrote", name)